# NP-completude: transformações polinomiais e reduções — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. Cada redução vira uma função $f$ que transforma uma instância do
problema conhecido $A$ numa instância do problema novo $B$. Os testes automáticos conferem as duas
direções da prova: a resposta de $f(x)$ é igual à de $x$ (comparando com a força bruta), e um
certificado de $f(x)$ volta a ser um certificado de $x$. A redução para clique é escrita em C, com
`gcc` chamado a partir do notebook; as demais ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- seguir a **receita** de uma redução: construir $f$, transportar certificados nos dois sentidos e conferir que $f$ é polinomial;
- implementar $\textsc{SAT} \le_P \textsc{3-SAT}$ por **substituição local** (corrente de variáveis novas);
- implementar em C $\textsc{3-SAT} \le_P \textsc{Clique}$ e **ler a atribuição** de volta a partir da clique;
- implementar $\textsc{Clique} \le_P \textsc{Cobertura por Vértices}$ pelo grafo complemento;
- implementar $\textsc{3-SAT} \le_P \textsc{Soma de Subconjuntos}$ com colunas decimais e medir por que os números ficam **exponencialmente grandes**;
- escrever sozinho a redução $\textsc{Soma de Subconjuntos} \le_P \textsc{Partição}$ (desafio).

**Formato das fórmulas.** Uma fórmula em FNC é uma lista de cláusulas; cada cláusula é uma lista de
inteiros não nulos: `3` é $x_3$ e `-3` é $\lnot x_3$. Uma atribuição é uma lista `[None, x1, ..., xn]`
de booleanos (a posição 0 não é usada), como na aula passada.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools, time

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor inteiro de 'nome=valor' da saída."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            return int(tok.split('=', 1)[1])
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def matriz(n, arestas):
    adj = [[False] * n for _ in range(n)]
    for u, v in arestas:
        adj[u][v] = adj[v][u] = True
    return adj

def grafo_aleatorio(n, p, rng):
    return [(u, v) for u in range(n) for v in range(u + 1, n) if rng.random() < p]

# ---- fórmulas ----
def num_vars(clausulas):
    return max((abs(l) for c in clausulas for l in c), default=0)

def satisfaz(clausulas, atrib):
    """A atribuição [None, x1, ..., xn] satisfaz todas as cláusulas?"""
    return all(any(atrib[l] if l > 0 else not atrib[-l] for l in c) for c in clausulas)

def sat_bruta(n, clausulas):
    """Força bruta com poda: devolve uma atribuição que satisfaz as cláusulas, ou None."""
    atrib = [None] * (n + 1)
    def valor(l):                       # True, False ou None (variável ainda livre)
        v = atrib[abs(l)]
        return v if v is None or l > 0 else not v
    def morta(c):                       # todos os literais já são falsos
        return all(valor(l) is False for l in c)
    def tenta(i):
        if any(morta(c) for c in clausulas):
            return False
        if i > n:
            return True
        for b in (False, True):
            atrib[i] = b
            if tenta(i + 1):
                return True
        atrib[i] = None
        return False
    return list(atrib) if tenta(1) else None

def fnc_aleatoria(n, m, rng, tam=(1, 5)):
    """m cláusulas com variáveis distintas em cada cláusula e tamanhos em [tam[0], tam[1]]."""
    return [[v * rng.choice((1, -1)) for v in rng.sample(range(1, n + 1), rng.randint(tam[0], min(tam[1], n)))]
            for _ in range(m)]

def tres_sat_aleatorio(n, m, rng):
    return fnc_aleatoria(n, m, rng, tam=(3, 3))

# Grafo condutor da aula passada: hexágono 0-1-2-3-4-5-0 com a corda 0-2.
HEXA = (6, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (0, 2)])

In [ ]:
# Desenho de grafos pequenos: vértices destacados em laranja, arestas destacadas grossas.
import matplotlib.pyplot as plt

def desenha(n, arestas, pos=None, rotulos=None, vdest=(), adest=(), titulo='', ax=None):
    if pos is None:
        pos = {v: (math.cos(math.pi / 2 + 2 * math.pi * v / n),
                   math.sin(math.pi / 2 + 2 * math.pi * v / n)) for v in range(n)}
    adest = {frozenset(a) for a in adest}
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3))
    for u, v in arestas:
        d = frozenset((u, v)) in adest
        ax.plot(*zip(pos[u], pos[v]), color='darkorange' if d else 'lightgray',
                lw=3 if d else 1, zorder=1 + d)
    for v, (x, y) in pos.items():
        ax.scatter(x, y, s=420, zorder=3, edgecolor='steelblue',
                   color='navajowhite' if v in vdest else 'lightsteelblue')
        ax.text(x, y, str(v) if rotulos is None else rotulos[v], ha='center', va='center',
                fontsize=8, zorder=4)
    ax.set_title(titulo, fontsize=10); ax.axis('off'); ax.set_aspect('equal'); ax.margins(0.12)
    return ax

## 1. A receita em código: $\textsc{Conjunto Independente} \le_P \textsc{Cobertura}$

Uma redução tem três partes que dá para programar:

1. a **transformação** $f$, que recebe só a instância $x$ (nunca a resposta);
2. a **ida** ($\Rightarrow$): um certificado de $x$ vira um certificado de $f(x)$;
3. a **volta** ($\Leftarrow$): um certificado de $f(x)$ vira um certificado de $x$.

No exemplo da aula, $f(G, k) = (G, n - k)$ e o certificado é traduzido por $S \mapsto V \setminus S$
nos dois sentidos. A célula abaixo implementa as três partes e testa contra a força bruta. Use-a
como modelo para os exercícios.

In [ ]:
# Exemplo: a receita para Conjunto Independente <=_P Cobertura por Vértices.

def e_independente(n, arestas, S):
    S = set(S)
    return all(not (u in S and v in S) for u, v in arestas)

def e_cobertura(n, arestas, C):
    C = set(C)
    return all(u in C or v in C for u, v in arestas)

def busca(n, k, teste):
    """Força bruta: algum subconjunto de k vértices passa no teste? Devolve-o ou None."""
    for S in itertools.combinations(range(n), k):
        if teste(S):
            return list(S)
    return None

# 1. transformação: só olha a instância
def indep_para_cobertura(n, arestas, k):
    return n, list(arestas), n - k

# 2 e 3. tradução de certificados (aqui a mesma nos dois sentidos)
def complemento_de(n, S):
    return sorted(set(range(n)) - set(S))

n, A = HEXA
S = [1, 3, 5]
n2, A2, k2 = indep_para_cobertura(n, A, len(S))
C = complemento_de(n, S)
print(f'independente {S} -> cobertura {C} de tamanho {k2}:', e_cobertura(n2, A2, C))

# Teste das duas direções em grafos aleatórios
rng = random.Random(0)
for _ in range(300):
    n = rng.randint(1, 8); A = grafo_aleatorio(n, rng.random(), rng); k = rng.randint(0, n)
    S = busca(n, k, lambda S: e_independente(n, A, S))
    n2, A2, k2 = indep_para_cobertura(n, A, k)
    C = busca(n2, k2, lambda C: e_cobertura(n2, A2, C))
    assert (S is None) == (C is None)                                # mesma resposta
    if S is not None:
        assert e_cobertura(n2, A2, complemento_de(n, S))             # ida
        assert e_independente(n, A, complemento_de(n, C))            # volta
print('receita ok nos 300 testes')

desenha(*HEXA, vdest={0, 2, 4}, titulo='cobertura {0,2,4} = V - {1,3,5}')
plt.show()

## 2. $\textsc{SAT} \le_P \textsc{3-SAT}$: substituição local

Cada cláusula com $\ell$ literais vira um bloco de cláusulas com **exatamente 3** literais,
usando variáveis novas, exclusivas daquele bloco (numere-as a partir de $n + 1$):

| $\ell$ | cláusula | cláusulas criadas |
|---|---|---|
| 1 | $(a)$ | $(a \lor y_1 \lor y_2)(a \lor y_1 \lor \lnot y_2)(a \lor \lnot y_1 \lor y_2)(a \lor \lnot y_1 \lor \lnot y_2)$ |
| 2 | $(a \lor b)$ | $(a \lor b \lor y_1)(a \lor b \lor \lnot y_1)$ |
| 3 | $(a \lor b \lor d)$ | ela mesma |
| $\ell > 3$ | $(a_1 \lor \cdots \lor a_\ell)$ | $(a_1 \lor a_2 \lor y_1)(\lnot y_1 \lor a_3 \lor y_2)\cdots(\lnot y_{\ell-3} \lor a_{\ell-1} \lor a_\ell)$ |

A célula abaixo mostra a força bruta de SAT que os testes usam e confere, para a cláusula de 5
literais dos slides, que a corrente só fica insatisfeita quando todos os literais originais são
falsos.

In [ ]:
# Exemplo: a corrente da cláusula (x1 v x2 v -x3 v x4 v x5), com y1 = x6 e y2 = x7.
c = [1, 2, -3, 4, 5]
corrente = [[1, 2, 6], [-6, -3, 7], [-7, 4, 5]]

for bits in itertools.product([False, True], repeat=5):
    atrib = [None, *bits]
    original = satisfaz([c], atrib)
    # existe valor para y1, y2 que satisfaz a corrente?
    estende = any(satisfaz(corrente, atrib + [y1, y2]) for y1 in (False, True) for y2 in (False, True))
    assert original == estende
print('para as 32 atribuições: c satisfeita  <=>  a corrente pode ser satisfeita')
print('única atribuição que falha: x1=x2=x4=x5=0 e x3=1')

In [ ]:
# Exercício 1: SAT <=_P 3-SAT
# TODO: devolva (n2, clausulas3): o número total de variáveis (originais + novas) e a lista de
#       cláusulas com EXATAMENTE 3 literais de variáveis distintas, seguindo a tabela acima.
#       As variáveis novas recebem os números n+1, n+2, ...

def para_3sat(n, clausulas):
    # TODO: implemente aqui
    pass

# Teste automático
n2, F3 = para_3sat(3, [[1, -2], [-1, 2, 3], [-3, -1]])
assert all(len(c) == 3 and len({abs(l) for l in c}) == 3 for c in F3), 'Toda cláusula precisa de 3 variáveis distintas'
assert n2 == 3 + 2, 'A fórmula dos slides precisa de 2 variáveis novas (uma por cláusula de 2 literais)'
n2, F3 = para_3sat(5, [[1, 2, -3, 4, 5]])
assert sorted(map(sorted, F3)) == sorted(map(sorted, [[1, 2, 6], [-6, -3, 7], [-7, 4, 5]])), 'Verifique a corrente'
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 6)
    F = fnc_aleatoria(n, rng.randint(1, 6), rng, tam=(1, 6))
    n2, F3 = para_3sat(n, F)
    assert all(len(c) == 3 and len({abs(l) for l in c}) == 3 for c in F3), f'cláusula inválida em {F3}'
    assert num_vars(F3) <= n2 and sum(map(len, F3)) <= 12 * sum(map(len, F)), 'a fórmula cresceu demais'
    a, a3 = sat_bruta(n, F), sat_bruta(n2, F3)
    assert (a is None) == (a3 is None), f'respostas diferentes para {F}'
    if a3 is not None:
        assert satisfaz(F, a3[:n + 1]), 'volta: as variáveis originais deveriam satisfazer F'
print('Exercício 1 ok!')

**Para pensar.** Por que a mesma ideia não reduz SAT a **2-SAT**? Tente quebrar
$(a \lor b \lor d)$ em cláusulas de 2 literais com uma variável nova $y$ e veja o que dá errado.
(2-SAT está em P, então, se P $\ne$ NP, nenhuma redução assim pode existir.)

## 3. $\textsc{3-SAT} \le_P \textsc{Clique}$: da lógica para grafos

Para $\varphi = C_1 \land \cdots \land C_k$, o grafo tem um vértice para cada literal de cada
cláusula (o vértice $3r + i$ é o $i$-ésimo literal de $C_r$) e uma aresta entre dois vértices quando
eles estão em **cláusulas diferentes** e **não são complementares**. Então $\varphi$ é satisfazível
se e só se o grafo tem clique de tamanho $k$.

A célula abaixo monta o grafo da fórmula dos slides (Cormen et al., Figura 34.14) em Python e
desenha a clique que corresponde a $x_2 = 0$, $x_3 = 1$.

In [ ]:
# Exemplo: o grafo da fórmula (x1 v -x2 v -x3)(-x1 v x2 v x3)(x1 v x2 v x3).
PHI = [[1, -2, -3], [-1, 2, 3], [1, 2, 3]]

def grafo_clausulas_py(clausulas):
    lit = [l for c in clausulas for l in c]
    N = len(lit)
    arestas = [(u, v) for u in range(N) for v in range(u + 1, N)
               if u // 3 != v // 3 and lit[u] != -lit[v]]
    return N, arestas, lit

N, arestas, lit = grafo_clausulas_py(PHI)
print(f'{N} vértices e {len(arestas)} arestas')

nome = lambda l: ('¬' if l < 0 else '') + f'x{abs(l)}'
pos = {0: (-1, 1.7), 1: (0, 1.7), 2: (1, 1.7),                 # C1 em cima
       3: (-2.3, 0.6), 4: (-2.3, -0.3), 5: (-2.3, -1.2),       # C2 à esquerda
       6: (2.3, 0.6), 7: (2.3, -0.3), 8: (2.3, -1.2)}          # C3 à direita
clique = [1, 5, 8]                                             # ¬x2 em C1, x3 em C2 e em C3
_, ax = plt.subplots(figsize=(5, 3.5))
desenha(N, arestas, pos=pos, rotulos=[nome(l) for l in lit], vdest=clique,
        adest=list(itertools.combinations(clique, 2)), titulo='clique de tamanho 3', ax=ax)
plt.show()

### Exercício 2 (em C)

Complete `montaGrafo` no programa abaixo. O resto já está pronto: a leitura da fórmula (no formato
`n m` e depois $m$ cláusulas de 3 literais, cada uma terminada por `0`), o decisor de clique por força
bruta da aula passada e a impressão da clique encontrada. Depois, a célula de teste lê a clique e
**reconstrói a atribuição** (a volta da prova).

In [ ]:
%%writefile src/sat_clique.c
/* Exercicio 2: 3-SAT <=_P Clique (Cormen et al., Teorema 34.11).
   Entrada: "n m" e m clausulas de 3 literais (+i / -i), cada uma terminada por 0.
   Saida: "arestas=... clique=1" e os vertices da clique de tamanho m, ou "clique=0".
   TODO: complete montaGrafo(). */
#include <stdio.h>

#define MAXM 8
#define MAXN (3 * MAXM)
int n;                  /* numero de vertices do grafo: 3m */
int lit[MAXN];          /* lit[3r + i] = i-esimo literal da clausula r */
int S[MAXN];            /* a clique encontrada */

/* Ha clique de tamanho k entre os vertices v..n-1, dado que S[0..t-1] ja e clique?
   Forca bruta com poda simples (decisor da aula passada). */
int estende( int a[][MAXN], int t, int v, int k) {
   if (t == k) return 1;
   if (v == n || t + (n - v) < k) return 0;
   int ok = 1;
   for (int i = 0; i < t && ok; ++i) if (!a[S[i]][v]) ok = 0;
   if (ok) { S[t] = v;  if (estende( a, t + 1, v + 1, k)) return 1; }
   return estende( a, t, v + 1, k);
}

/* Preenche a[][] com a matriz de adjacencias do grafo das clausulas: aresta entre u e v
   se estao em clausulas diferentes e os literais nao sao complementares. */
void montaGrafo( int a[][MAXN]) {
   /* TODO */
   for (int u = 0; u < n; ++u)
      for (int v = 0; v < n; ++v) a[u][v] = 0;
}

int main( void) {
   int nv, m, a[MAXN][MAXN];
   if (scanf( "%d %d", &nv, &m) != 2 || m > MAXM) return 1;
   for (int r = 0; r < m; ++r) {
      int zero;
      for (int i = 0; i < 3; ++i) if (scanf( "%d", &lit[3 * r + i]) != 1) return 1;
      if (scanf( "%d", &zero) != 1 || zero != 0) return 1;
   }
   n = 3 * m;
   montaGrafo( a);                                  /* a reducao f */
   int arestas = 0;
   for (int u = 0; u < n; ++u)
      for (int v = u + 1; v < n; ++v) arestas += a[u][v];
   printf( "arestas=%d ", arestas);
   if (estende( a, 0, 0, m)) {
      printf( "clique=1\n");
      for (int i = 0; i < m; ++i) printf( "%d ", S[i]);
      printf( "\n");
   } else
      printf( "clique=0\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 2

def sat_clique_c(n, clausulas):
    """Roda o programa em C; devolve (número de arestas, lista de vértices da clique ou None)."""
    entrada = f'{n} {len(clausulas)}\n' + ''.join(' '.join(map(str, c)) + ' 0\n' for c in clausulas)
    saida = rodar('sat_clique.c', entrada=entrada, mostrar=False)
    linhas = saida.split('\n')
    clique = list(map(int, linhas[1].split())) if campo(saida, 'clique') == 1 else None
    return campo(saida, 'arestas'), clique

def atribuicao_da_clique(n, clausulas, clique):
    """A volta da prova: torna verdadeiros os literais da clique; as demais variáveis ficam falsas."""
    lit = [l for c in clausulas for l in c]
    atrib = [None] + [False] * n
    for v in clique:
        atrib[abs(lit[v])] = lit[v] > 0
    return atrib

ar, cl = sat_clique_c(3, PHI)
assert ar == 21, f'A fórmula dos slides deve gerar 21 arestas, não {ar}'
assert cl is not None and satisfaz(PHI, atribuicao_da_clique(3, PHI, cl)), 'Verifique a clique da fórmula dos slides'
print('fórmula dos slides: 21 arestas, clique', cl)
TODAS = [[a * 1, b * 2, c * 3] for a in (1, -1) for b in (1, -1) for c in (1, -1)]   # insatisfazível
ar, cl = sat_clique_c(3, TODAS)
assert cl is None, 'As 8 cláusulas sobre x1, x2, x3 são insatisfazíveis: não pode haver clique de tamanho 8'
print(f'8 cláusulas insatisfazíveis: {ar} arestas e nenhuma clique de tamanho 8')
rng = random.Random(2026)
sat = insat = 0
for _ in range(150):
    n = rng.randint(3, 5)
    F = tres_sat_aleatorio(n, rng.randint(1, 8), rng)
    _, cl = sat_clique_c(n, F)
    a = sat_bruta(n, F)
    assert (cl is None) == (a is None), f'respostas diferentes para {F}'
    if cl is not None:
        assert satisfaz(F, atribuicao_da_clique(n, F, cl)), f'a clique não dá atribuição válida em {F}'
        sat += 1
    else:
        insat += 1
print(f'Exercício 2 ok! ({sat} fórmulas satisfazíveis, {insat} insatisfazíveis)')

## 4. $\textsc{Clique} \le_P \textsc{Cobertura por Vértices}$

$S$ é clique em $G$ se e só se $V \setminus S$ é cobertura do complemento $\overline{G}$. Então
$f(G, k) = (\overline{G}, n - k)$. Atenção ao limiar: $k$ vira $n - k$.

### Exercício 3

Implemente as três partes da receita, como no exemplo da seção 1:

- `clique_para_cobertura(n, arestas, k)` devolve `(n, arestas_do_complemento, n - k)`;
- `clique_para_cert_cobertura(n, S)` traduz uma clique de $G$ numa cobertura de $\overline{G}$ (ida);
- `cobertura_para_cert_clique(n, C)` traduz uma cobertura de $\overline{G}$ numa clique de $G$ (volta).

In [ ]:
# Exercício 3: Clique <=_P Cobertura por Vértices
# TODO: implemente as três funções descritas acima.

def clique_para_cobertura(n, arestas, k):
    # TODO: implemente aqui
    pass

def clique_para_cert_cobertura(n, S):
    # TODO: implemente aqui
    pass

def cobertura_para_cert_clique(n, C):
    # TODO: implemente aqui
    pass

# Teste automático
def e_clique(n, arestas, S):
    adj = matriz(n, arestas)
    return all(adj[u][v] for u, v in itertools.combinations(S, 2))

n2, A2, k2 = clique_para_cobertura(*HEXA, 3)
assert sorted(map(tuple, map(sorted, A2))) == [(0, 3), (0, 4), (1, 3), (1, 4), (1, 5), (2, 4), (2, 5), (3, 5)], 'Verifique o complemento'
assert k2 == 3 and sorted(clique_para_cert_cobertura(6, [0, 1, 2])) == [3, 4, 5], 'Verifique (slides: cobertura {3,4,5})'
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 8); A = grafo_aleatorio(n, rng.random(), rng); k = rng.randint(0, n)
    S = busca(n, k, lambda S: e_clique(n, A, S))
    n2, A2, k2 = clique_para_cobertura(n, A, k)
    C = busca(n2, k2, lambda C: e_cobertura(n2, A2, C))
    assert (S is None) == (C is None), f'respostas diferentes em n={n}, {A}, k={k}'
    if S is not None:
        assert e_cobertura(n2, A2, clique_para_cert_cobertura(n, S)), 'ida falhou'
        assert e_clique(n, A, cobertura_para_cert_clique(n, C)), 'volta falhou'
print('Exercício 3 ok!')

Encadeando os exercícios 1, 2 e 3, qualquer fórmula SAT vira uma instância de cobertura por
vértices em tempo polinomial: $\textsc{SAT} \le_P \textsc{3-SAT} \le_P \textsc{Clique} \le_P
\textsc{Cobertura}$. A composição de reduções polinomiais é polinomial.

## 5. $\textsc{3-SAT} \le_P \textsc{Soma de Subconjuntos}$: da lógica para números

Cada número tem $n + k$ dígitos decimais: primeiro uma coluna por variável, depois uma por cláusula.

- $v_i$: dígito 1 na coluna $x_i$ e nas colunas das cláusulas que contêm $x_i$;
- $v'_i$: dígito 1 na coluna $x_i$ e nas colunas das cláusulas que contêm $\lnot x_i$;
- $s_j$ e $s'_j$: dígito 1 e dígito 2 na coluna $C_j$;
- alvo $t$: 1 em cada coluna de variável e 4 em cada coluna de cláusula.

Como cada coluna soma no máximo 6, não há vai-um, e a soma é $t$ se e só se cada coluna bate.

Para testar, precisamos de um decisor de soma de subconjuntos que aguente números enormes. A PD da
aula passada usaria uma tabela com $t + 1$ colunas, e aqui $t \approx 10^{n+k}$. A célula abaixo traz
um decisor por **encontro no meio** (*meet-in-the-middle*): divide os $m$ números em duas metades,
lista as $2^{m/2}$ somas de cada uma e procura um par que some $t$. Custa $O(2^{m/2})$:
exponencial em $m$, mas independente do tamanho dos números.

In [ ]:
# Ferramenta: soma de subconjuntos por encontro no meio, O(2^(m/2)) somas.

def _somas(numeros, deslocamento):
    """Dicionário soma -> índices, para todos os subconjuntos de numeros."""
    somas = {0: ()}
    for i, a in enumerate(numeros):
        for s, idx in list(somas.items()):
            somas.setdefault(s + a, idx + (deslocamento + i,))
    return somas

def soma_subconjunto(numeros, t):
    """Devolve uma lista de índices com soma t, ou None."""
    h = len(numeros) // 2
    esq, dir = _somas(numeros[:h], 0), _somas(numeros[h:], h)
    for s, idx in esq.items():
        if t - s in dir:
            return sorted(idx + dir[t - s])
    return None

print(soma_subconjunto([3, 5, 6, 7], 15))        # instância da aula passada
print(soma_subconjunto([3, 5, 6, 7], 2))

### Exercício 4

Implemente:

- `para_soma(n, clausulas)`, que devolve `(numeros, t)` com os números **nesta ordem**:
  $v_1, v'_1, v_2, v'_2, \ldots, v_n, v'_n, s_1, s'_1, \ldots, s_k, s'_k$;
- `atribuicao_da_soma(n, indices)`, a volta da prova: a partir dos índices escolhidos em `numeros`,
  devolva a atribuição `[None, x1, ..., xn]` ($x_i$ é verdadeiro se $v_i$ foi escolhido).

*Dica:* monte os dígitos numa lista e converta com `int(''.join(map(str, digitos)))`.

In [ ]:
# Exercício 4: 3-SAT <=_P Soma de Subconjuntos
# TODO: implemente as duas funções descritas acima.

def para_soma(n, clausulas):
    # TODO: implemente aqui
    pass

def atribuicao_da_soma(n, indices):
    # TODO: implemente aqui
    pass

# Teste automático
CLRS = [[1, -2, -3], [-1, -2, -3], [-1, -2, 3], [1, 2, 3]]       # Cormen et al., Figura 34.19
nums, t = para_soma(3, CLRS)
assert t == 1114444, f'O alvo dos slides é 1114444, não {t}'
assert nums == [1001001, 1000110, 100001, 101110, 10011, 11100,
                1000, 2000, 100, 200, 10, 20, 1, 2], 'Verifique a tabela de dígitos (compare com os slides)'
assert soma_subconjunto(nums, t) is not None
rng = random.Random(11)
sat = insat = 0
for _ in range(120):
    n = rng.randint(3, 4)
    F = tres_sat_aleatorio(n, rng.randint(1, 10), rng)
    nums, t = para_soma(n, F)
    assert len(nums) == 2 * n + 2 * len(F) and len(str(t)) == n + len(F), 'tamanho errado'
    idx = soma_subconjunto(nums, t)
    a = sat_bruta(n, F)
    assert (idx is None) == (a is None), f'respostas diferentes para {F}'
    if idx is not None:
        assert satisfaz(F, atribuicao_da_soma(n, idx)), f'volta falhou em {F}'
        sat += 1
    else:
        insat += 1
print(f'Exercício 4 ok! ({sat} satisfazíveis, {insat} insatisfazíveis)')

In [ ]:
# Visualização: tamanho da instância (dígitos) x valor do alvo t (células da PD da aula passada).
# Fórmulas 3-SAT com n variáveis e k = 4n cláusulas.
rng = random.Random(1)
ns, digitos, celulas = [], [], []
for n in range(3, 16):
    F = tres_sat_aleatorio(n, 4 * n, rng)
    nums, t = para_soma(n, F)
    ns.append(n)
    digitos.append(sum(len(str(x)) for x in nums))           # tamanho da entrada
    celulas.append(len(nums) * (t + 1))                       # tabela (m+1)(t+1) da PD

plt.figure(figsize=(5.5, 3))
plt.semilogy(ns, digitos, 'o-', label='tamanho da instância (dígitos)')
plt.semilogy(ns, celulas, 's-', label='células da PD Θ(m·t)')
plt.xlabel('variáveis n (k = 4n cláusulas)'); plt.ylabel('escala log'); plt.grid(alpha=0.3)
plt.legend(); plt.title('Tamanho polinomial, valor exponencial'); plt.show()
print('n = 15:', digitos[-1], 'dígitos na entrada; a tabela da PD teria', f'{celulas[-1]:.1e}', 'células')

O gráfico explica por que a redução não prova P $=$ NP: ela produz números com $O(n + k)$ dígitos
(tamanho polinomial), mas de **valor** $\approx 10^{n+k}$. A PD pseudopolinomial, polinomial no valor de
$t$, fica exponencial justamente nas instâncias que vêm do 3-SAT. É a NP-completude **fraca** da
aula passada.

## Desafio Final: $\textsc{Soma de Subconjuntos} \le_P \textsc{Partição}$

$\textsc{Partição}$: dada uma lista de inteiros positivos, dá para dividi-la em duas partes de mesma
soma? Escreva a redução a partir de $\textsc{Soma de Subconjuntos}$ e as duas traduções de
certificado:

- `soma_para_particao(numeros, t)` devolve a lista de entrada de $\textsc{Partição}$;
- `soma_para_cert_particao(numeros, t, indices)` (ida): dado um subconjunto de soma $t$, devolve os
  índices (na lista nova) de uma das duas metades;
- `particao_para_cert_soma(numeros, t, metade)` (volta): dados os índices de uma metade na lista
  nova, devolve índices de `numeros` com soma $t$.

*Dicas:* seja $\sigma$ a soma de `numeros`. Acrescente dois números escolhidos de modo que eles
**não possam** ficar na mesma metade e que a metade de um deles precise completar exatamente $t$ com
os números originais (o slide “Exemplo: Soma de Subconjuntos $\le_P$ Partição” tem a construção).
Se $t > \sigma$, a resposta é não: devolva uma instância “não” fixa, como `[1, 2]`.

Antes de programar, escreva no papel os seis passos da receita: é exatamente o que a lista de
reduções pede.

In [ ]:
# Desafio: Soma de Subconjuntos <=_P Partição
# TODO: resolva o desafio abaixo

def particao_bruta(lista):
    """Força bruta: índices de uma metade com soma total/2, ou None."""
    total = sum(lista)
    if total % 2:
        return None
    for k in range(len(lista) + 1):
        for S in itertools.combinations(range(len(lista)), k):
            if sum(lista[i] for i in S) == total // 2:
                return list(S)
    return None

def soma_para_particao(numeros, t):
    # Sua solução aqui
    pass

def soma_para_cert_particao(numeros, t, indices):
    # Sua solução aqui
    pass

def particao_para_cert_soma(numeros, t, metade):
    # Sua solução aqui
    pass

# Teste automático (descomente quando terminar)
# P = soma_para_particao([3, 5, 6, 7], 15)
# assert sorted(P) == [3, 5, 6, 7, 27, 36]                     # slides: b = 36, b' = 27
# rng = random.Random(8)
# for _ in range(300):
#     nums = [rng.randint(1, 15) for _ in range(rng.randint(1, 7))]
#     t = rng.randint(0, sum(nums) + 5)
#     P = soma_para_particao(nums, t)
#     assert all(x > 0 for x in P)
#     s, p = soma_subconjunto(nums, t), particao_bruta(P)
#     assert (s is None) == (p is None)
#     if s is not None:
#         m = soma_para_cert_particao(nums, t, s)
#         assert 2 * sum(P[i] for i in m) == sum(P)
#         idx = particao_para_cert_soma(nums, t, p)
#         assert len(set(idx)) == len(idx) and sum(nums[i] for i in idx) == t
# print('Desafio ok!')

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 34 (§34.4 Lema 34.8 e 3-CNF-SAT; §34.5.1 clique, Teorema 34.11 e Figura 34.14; §34.5.2 cobertura por vértices, Teorema 34.12; §34.5.5 soma de subconjuntos, Teorema 34.15 e Figura 34.19). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- GAREY, M. R.; JOHNSON, D. S. *Computers and Intractability*. W. H. Freeman, 1979. §3.1.1 (SAT para 3SAT, Teorema 3.1), §3.1.3 (Lema 3.1), §3.1.5 (partição), §3.2 (técnicas de prova).
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §8.1, §8.2, §8.4 e §8.8.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §11.3.
- TOSCANI, L. V.; VELOSO, P. A. S. *Complexidade de Algoritmos*. 3. ed. Bookman, 2012.
- KARP, R. M. Reducibility among Combinatorial Problems. In: *Complexity of Computer Computations*. Plenum, 1972. p. 85–103.